In [ ]:
# Cell 1: Import dependencies and resolve local config paths
from pathlib import Path
import ast
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = cfg.get('datasets', {}).get('rml2018', {})
    h5_path = Path(dcfg.get('hdf5', '/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5'))
    classes_path = Path(dcfg.get('classes', '/scratch/rameyjm7/datasets/RML2018/classes.txt'))
    classes_fixed_path = Path(dcfg.get('classes_fixed', '/scratch/rameyjm7/datasets/RML2018/classes-fixed.txt'))
else:
    h5_path = Path('/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5')
    classes_path = Path('/scratch/rameyjm7/datasets/RML2018/classes.txt')
    classes_fixed_path = Path('/scratch/rameyjm7/datasets/RML2018/classes-fixed.txt')

model_path = project_root / 'models' / 'rml2018' / 'rml2018_lstm_rnn.keras'
print('RML2018 dataset:', h5_path)
print('RML2018 model:', model_path)
assert h5_path.exists(), f'Missing dataset: {h5_path}'
assert classes_path.exists(), f'Missing classes file: {classes_path}'
assert classes_fixed_path.exists(), f'Missing classes-fixed file: {classes_fixed_path}'
assert model_path.exists(), f'Missing model: {model_path}'

In [ ]:
# Cell 2: Load a class-balanced highest-SNR evaluation slice

def parse_classes(path: Path):
    text = path.read_text()
    match = re.search(r'classes\s*=\s*(\[[\s\S]*?\])', text)
    if not match:
        raise ValueError(f'Could not parse classes from {path}')
    return ast.literal_eval(match.group(1))

classes_orig = parse_classes(classes_path)
classes_fixed = parse_classes(classes_fixed_path)
remap = np.array([classes_fixed.index(c) for c in classes_orig], dtype=np.int64)

with h5py.File(h5_path, 'r') as h5:
    X = h5['X']
    Y = h5['Y']
    Z = h5['Z']

    snr = Z[:, 0]
    max_snr = int(np.max(snr))
    max_idx = np.where(snr == max_snr)[0]
    y_max = np.argmax(Y[max_idx], axis=1)

    rng = np.random.default_rng(42)
    picked = []
    for cls in np.unique(y_max):
        cls_idx = max_idx[y_max == cls]
        k = min(200, len(cls_idx))
        picked.extend(rng.choice(cls_idx, size=k, replace=False).tolist())

    picked = np.array(sorted(picked), dtype=np.int64)
    x_iq = np.asarray(X[picked], dtype=np.float32)
    y_orig = np.argmax(np.asarray(Y[picked]), axis=1).astype(np.int64)
    snr_vals = np.asarray(Z[picked, 0], dtype=np.float32)

y_true = remap[y_orig]
snr_ch = np.repeat(snr_vals[:, None, None], x_iq.shape[1], axis=1)
X_eval = np.concatenate([x_iq, snr_ch], axis=2).astype(np.float32)

print('X_eval shape:', X_eval.shape)
print('max_snr used:', max_snr)
print('class count:', len(classes_fixed))

In [ ]:
# Cell 3: Run model inference and print metrics
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_eval, verbose=0), axis=1)

acc = accuracy_score(y_true, y_pred)
print(f'RML2018 evaluation accuracy: {acc:.4f}')
print(classification_report(y_true, y_pred, target_names=classes_fixed, zero_division=0))

In [ ]:
# Cell 4: Plot confusion matrix for RML2018
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes_fixed)))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues')
plt.title('RML2018 Confusion Matrix (class index view)')
plt.xlabel('Predicted class index')
plt.ylabel('True class index')
plt.tight_layout()
plt.show()